# NB10 — Step 5: phoneme n-gram LM rescoring (toward 0.70+)

Our 0.667 model decodes **greedily** (argmax per frame, each phoneme chosen
independently). A phoneme **n-gram language model** adds knowledge of which
phoneme *sequences* are plausible in Quranic Arabic, fixing DECODING slips
(e.g. the رَبِّ word-boundary artifact, the إِيَّاكَ glottal/glide confusion)
without retraining the acoustic model.

### The danger we tread carefully around
An LM can **over-correct**: pull a genuinely mispronounced sequence back toward
the 'expected' one because it is more probable in the language. For an MDD task
that means **hiding a real mispronunciation** — the worst outcome, and the exact
failure mode that killed Whisper. So:

> **We tune the LM weight with RECALL as a guard rail.** Recall = catch-rate of
> real errors. We REJECT any LM setting that lowers recall vs the 0.667 baseline,
> even if F1 rises. A missed mispronunciation is not an acceptable trade.

Why a **general** phoneme LM (trained on the canonical Quran phoneme corpus) and
NOT a per-verse constrained decode: the general LM learns phonotactics, not 'what
this verse should be', so it cannot silently rewrite a wrong recitation into the
right answer. We deliberately avoid the constrained-decode approach for that
reason.

Everything is judged by the **official IqraEval scorer** (same as NB5).

In [ ]:
# ---- Installs (minimal, numpy-safe) -----------------------------------------
# Step 5 now uses a PURE-PYTHON phoneme n-gram LM (built later in the notebook),
# so there are NO compiled LM dependencies (no kenlm, no pyctcdecode) and nothing
# can downgrade numpy / break pandas. We only need the model + data + scorer libs.
import numpy, subprocess, sys
print("numpy before:", numpy.__version__)

!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa pandas

# Official scorer:
!git clone -q https://github.com/Iqra-Eval/interspeech_IqraEval.git || echo 'already cloned'
!ls interspeech_IqraEval/mdd_eval

import numpy as _np_check
print("numpy after:", _np_check.__version__)
assert _np_check.__version__.startswith("2."), "numpy changed unexpectedly; restart."
print("OK: minimal deps, pure-Python LM (no kenlm packaging risk).")

In [ ]:
import os, sys, subprocess, re
import numpy as np, torch, pandas as pd

try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
    print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN — need it for the gated GT truth file:", e)

# The 0.667 real-error model.
MODEL_DIR = "/kaggle/input/datasets/qasimmaajid/model-3/w2vbert-quran-phoneme-real/final"   # EDIT
SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVAL = "interspeech_IqraEval/mdd_eval"
print("device:", DEVICE, "| model exists:", os.path.isdir(MODEL_DIR))

In [ ]:
from transformers import AutoModelForCTC, AutoFeatureExtractor, Wav2Vec2CTCTokenizer

model = AutoModelForCTC.from_pretrained(MODEL_DIR).to(DEVICE).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(MODEL_DIR)

# Vocab in INDEX order — pyctcdecode needs the label list aligned to logit columns.
vocab_size = model.config.vocab_size
id2tok = {i: t for t, i in tokenizer.get_vocab().items()}
labels = [id2tok.get(i, "") for i in range(vocab_size)]
blank_id = tokenizer.pad_token_id
DROP = {"<sil>", "sil", "<pad>", "<unk>", "<s>", "</s>", "|", ""}
print("vocab size:", vocab_size, "| blank id:", blank_id)
print("first labels:", labels[:8])

In [ ]:
# ---- Cache per-clip LOGITS once (greedy + every beam setting reuse these) ----
# Running the acoustic model is the expensive part; we do it ONE time and then
# sweep LM settings cheaply on the cached logits.
from datasets import load_dataset, Audio
qmb = load_dataset("IqraEval/QuranMB.v2", split="test")
qmb = qmb.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
print(qmb)

logits_cache, id_list = [], []
for k, ex in enumerate(qmb):
    feats = feature_extractor(np.asarray(ex["audio"]["array"], dtype=np.float32),
                              sampling_rate=SAMPLING_RATE, return_tensors="pt")
    feats = {kk: vv.to(DEVICE) for kk, vv in feats.items()}
    with torch.no_grad():
        lg = model(**feats).logits[0].cpu().numpy().astype(np.float32)
    logits_cache.append(lg)
    id_list.append(ex["ID"])
    if (k+1) % 200 == 0:
        print(f"  {k+1}/{len(qmb)} logits cached")
print("cached logits for", len(logits_cache), "clips")

In [ ]:
# ---- Build the TRUTH csv (gated GT) and the official-scorer helper -----------
gt = load_dataset("IqraEval/IqraEval_Test_GT", split="test")
pd.DataFrame({"ID": gt["ID"], "Reference_phn": gt["Reference_phn"],
              "Annotation_phn": gt["Annotation_phn"]}).to_csv("truth.csv", index=False)
print("wrote truth.csv")

def official_score(pred_df, tag=""):
    """Run the OFFICIAL scorer; return (f1, precision, recall)."""
    pred_df.to_csv("pred.csv", index=False)
    subprocess.run([sys.executable, f"{EVAL}/align_data.py",
                    "--truth-file", "truth.csv", "--pred-file", "pred.csv",
                    "--output-dir", "aligned"], capture_output=True, text=True)
    r = subprocess.run([sys.executable, f"{EVAL}/ins_del_cor_sub_analysis.py",
                        "--align-folder", "aligned/"], capture_output=True, text=True)
    out = r.stdout
    def grab(name):
        m = re.search(name + r":\s*([0-9.]+)", out)
        return float(m.group(1)) if m else float("nan")
    f1, p, rec = grab("f1-score"), grab("Precision"), grab("Recall")
    print(f"  [{tag}] F1={f1:.4f}  P={p:.4f}  R={rec:.4f}")
    return f1, p, rec

In [ ]:
# ---- BASELINE: greedy decode from the cached logits (must reproduce 0.667) ---
# Sanity check: the greedy path here should match NB5's 0.6666. If it doesn't,
# the label ordering or drop-set is off and we fix that BEFORE trusting any LM.
def greedy_decode(logits):
    ids = logits.argmax(-1).tolist()
    out, prev = [], None
    for i in ids:
        if i != prev and i != blank_id:
            sym = labels[i] if i < len(labels) else ""
            if sym not in DROP:
                out.append(sym)
        prev = i
    return " ".join(out)

greedy_df = pd.DataFrame({"ID": id_list,
                          "Prediction": [greedy_decode(lg) for lg in logits_cache]})
print("baseline greedy (expect ~0.6666):")
base_f1, base_p, base_r = official_score(greedy_df, tag="greedy")
print(f"\n>>> BASELINE  F1={base_f1:.4f}  Precision={base_p:.4f}  Recall={base_r:.4f}")
print("    Recall here is the GUARD RAIL: no LM setting may go below it.")

In [ ]:
# ---- Build the phoneme n-gram LM corpus from the CANONICAL references --------
# We train the LM on canonical phoneme sequences (the 'correct' way verses sound).
# This teaches general phonotactics. We use the GT Reference_phn (canonical) plus
# the train set's phoneme_ref so the LM sees plenty of well-formed sequences.
# NOTE: we do NOT include the annotated *mispronounced* sequences — the LM should
# model correct phonotactics; the acoustic model remains responsible for spotting
# deviations from the audio.
corpus_lines = []
# canonical references from the test GT (correct phonotactics for these verses)
corpus_lines += [str(s) for s in gt["Reference_phn"]]
# plus a big chunk of canonical training references for coverage
try:
    tr = load_dataset("IqraEval/Iqra_train", split="train")
    col = "phoneme_ref" if "phoneme_ref" in tr.column_names else None
    if col:
        corpus_lines += [str(s) for s in tr[col]]
    print("train canonical lines added:", 0 if not col else len(tr))
except Exception as e:
    print("train set not added (fine, GT refs alone work):", e)

# clean: keep only model-vocab phoneme tokens, drop sil/specials
vocab_syms = set(labels) - DROP
def clean_line(s):
    toks = [t for t in str(s).split() if t in vocab_syms]
    return " ".join(toks)
corpus_lines = [clean_line(s) for s in corpus_lines]
corpus_lines = [s for s in corpus_lines if s]
with open("phon_corpus.txt", "w") as f:
    f.write("\n".join(corpus_lines))
print("corpus lines:", len(corpus_lines))
print("sample:", corpus_lines[0][:120])

In [ ]:
# ---- Pure-Python phoneme n-gram LM (no compiled deps, can't fail to install) -
# kenlm's binary bindings kept failing to build on Kaggle (a packaging problem,
# not a modeling one). Our corpus is tiny (~73k short lines, vocab ~70), so a
# self-contained interpolated n-gram is entirely adequate and has ZERO install
# risk. Data-scientist's call: stop gambling GPU runs on a C++ dependency when a
# transparent Python LM does the same job at our scale.
import math
from collections import defaultdict

class NgramLM:
    # Interpolated n-gram with add-k smoothing. Scores log10 P(token | history).
    # Trained on canonical phoneme sequences -> models correct phonotactics
    # (general 'how Arabic sounds', NOT a specific verse).
    def __init__(self, order=5, k=0.1):
        self.order = order
        self.k = k
        self.counts = [defaultdict(lambda: defaultdict(int)) for _ in range(order)]
        self.ctx_tot = [defaultdict(int) for _ in range(order)]
        self.vocab = set()

    def train(self, lines):
        for ln in lines:
            toks = ["<s>"] + ln.split() + ["</s>"]
            self.vocab.update(toks)
            for n in range(self.order):
                for i in range(n, len(toks)):
                    ctx = tuple(toks[i-n:i])
                    self.counts[n][ctx][toks[i]] += 1
                    self.ctx_tot[n][ctx] += 1
        self.V = len(self.vocab)
        print("  LM trained: order=%d vocab=%d contexts=%d"
              % (self.order, self.V, sum(len(c) for c in self.counts)))

    def _p_order(self, n, ctx, w):
        c = self.counts[n][ctx].get(w, 0)
        tot = self.ctx_tot[n].get(ctx, 0)
        return (c + self.k) / (tot + self.k * self.V) if (tot or self.k) else 0.0

    def logp(self, w, history):
        # interpolate all orders, weighting longer matched context more
        p, wsum = 0.0, 0.0
        for n in range(self.order):
            if n > len(history):
                break
            ctx = tuple(history[len(history)-n:]) if n else tuple()
            weight = n + 1
            p += weight * self._p_order(n, ctx, w)
            wsum += weight
        p = p / wsum if wsum else 1e-10
        return math.log10(max(p, 1e-10))

    def score_seq(self, toks):
        # total log10 prob of a phoneme token list, with sentence boundaries
        s, hist = 0.0, []
        for w in list(toks) + ["</s>"]:
            s += self.logp(w, hist)
            hist.append(w)
        return s

ORDER = 5
lm = NgramLM(order=ORDER, k=0.1)
lm.train(corpus_lines)
LM_OK = lm.V > 0

# Sanity: a real canonical line should score HIGHER per-token than a shuffled
# version of itself (the LM must prefer well-formed phonotactics).
import random
_t = corpus_lines[0].split()
_shuf = _t[:]; random.shuffle(_shuf)
print("  sanity canonical/token=%.3f  shuffled/token=%.3f  (canonical should be higher)"
      % (lm.score_seq(_t)/max(len(_t),1), lm.score_seq(_shuf)/max(len(_shuf),1)))
print("LM ready:", LM_OK)

In [ ]:
# ---- N-best rescoring with the phoneme LM, alpha sweep ----------------------
# We avoid beam search (that needed kenlm). Instead, per clip we build a small
# set of CANDIDATE phoneme sequences from the cached logits, then re-rank them by
#     score = acoustic_logprob  +  alpha * LM_logprob
# and pick the best. alpha = LM weight (higher = more LM influence = MORE
# over-correction risk). We sweep alpha and the DECISION cell enforces
# recall >= baseline so we never trade away real-error detection.
#
# Candidate generation: at each collapsed CTC step we look at the top-2 phoneme
# alternatives; where they're a CLOSE call (small log-prob gap) we allow swapping
# to the runner-up. We expand a bounded number of the most-uncertain positions
# into variant sequences. This targets exactly the decoding slips an LM can fix
# (close calls), and leaves confident outputs untouched.
import numpy as np, itertools

def _softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def candidates_from_logits(logits, max_variants=16, margin=0.45, max_swap_pos=6):
    logp = np.log(np.clip(_softmax(logits), 1e-9, 1.0))
    # collapse CTC to emission steps, keeping top-1 and top-2 per step
    steps, prev = [], None
    for t in range(logp.shape[0]):
        order = np.argsort(logp[t])[::-1]
        top1 = int(order[0])
        if top1 != prev and top1 != blank_id:
            top2 = int(order[1])
            steps.append((top1, float(logp[t, top1]), top2, float(logp[t, top2])))
        prev = top1
    base = [(i1, l1) for (i1, l1, i2, l2) in steps]          # greedy path
    base_ac = sum(l for _, l in base)
    base_seq = [labels[i] for i, _ in base if labels[i] not in DROP]

    # uncertain positions: small gap to a REAL-phoneme runner-up -> swap candidates
    gaps = []
    for idx, (i1, l1, i2, l2) in enumerate(steps):
        if labels[i2] in DROP:
            continue
        if (l1 - l2) < margin:
            gaps.append((l1 - l2, idx))
    gaps.sort()
    swap_idx = [g[1] for g in gaps[:max_swap_pos]]

    cands = [(base_ac, base_seq)]
    combos = []
    for r in (1, 2):
        for combo in itertools.combinations(swap_idx, r):
            combos.append(combo)
            if len(combos) >= max_variants:
                break
        if len(combos) >= max_variants:
            break
    for combo in combos:
        seq_ids = list(base)
        for j in combo:
            i1, l1, i2, l2 = steps[j]
            seq_ids[j] = (i2, l2)
        ac = sum(l for _, l in seq_ids)
        toks = [labels[i] for i, _ in seq_ids if labels[i] not in DROP]
        cands.append((ac, toks))
    return cands

# Precompute candidates ONCE (independent of alpha) so the sweep is cheap.
print("building n-best candidates per clip (one-time)...")
all_cands = [candidates_from_logits(lg) for lg in logits_cache]
print("  avg candidates/clip:", round(np.mean([len(c) for c in all_cands]), 1))

def rescore_all(alpha):
    preds = []
    for cands in all_cands:
        best_seq, best_score = None, -1e18
        for ac, toks in cands:
            lm_s = lm.score_seq(toks) if toks else -1e9
            score = ac + alpha * lm_s
            if score > best_score:
                best_score, best_seq = score, toks
        preds.append(" ".join(best_seq or []))
    return pd.DataFrame({"ID": id_list, "Prediction": preds})

results = [("greedy", 0.0, 0.0, base_f1, base_p, base_r)]
if not LM_OK:
    print(">>> LM unavailable; keeping greedy 0.667.")
else:
    print(f"\nGUARD RAIL: recall must stay >= {base_r:.4f}. Baseline F1={base_f1:.4f}.\n")
    for alpha in [0.5, 1.0, 2.0, 4.0, 7.0, 12.0]:
        f1, p, rec = official_score(rescore_all(alpha), tag=f"alpha={alpha}")
        results.append((f"a={alpha}", alpha, 0.0, f1, p, rec))

In [ ]:
# ---- DECISION: best setting that does NOT sacrifice recall ------------------
# Rule (false-negative-averse): among settings with recall >= baseline recall,
# take the highest F1. We report the gain honestly even when it is tiny — a small
# recall-safe improvement is still a real (if minor) win; a recall-COSTING F1
# bump is rejected outright.
REC_EPS = 0.0005   # recall tolerance (scorer noise); gains reported as-is
safe = [r for r in results if r[5] >= base_r - REC_EPS]
best_safe = max(safe, key=lambda r: r[3]) if safe else ("greedy",0,0,base_f1,base_p,base_r)
best_any  = max(results, key=lambda r: r[3])

print("all settings (tag, alpha, F1, P, R):")
for r in results:
    flag = "  <- recall-safe" if r[5] >= base_r - REC_EPS else "  (drops recall)"
    print(f"  {r[0]:10s} a={r[1]:.2f}  F1={r[3]:.4f} P={r[4]:.4f} R={r[5]:.4f}{flag}")

gain = best_safe[3] - base_f1
print(f"\nBaseline (greedy):           F1={base_f1:.4f}  P={base_p:.4f}  R={base_r:.4f}")
print(f"Best recall-safe setting:    {best_safe[0]}  F1={best_safe[3]:.4f}  "
      f"P={best_safe[4]:.4f}  R={best_safe[5]:.4f}")
print(f"F1 gain over baseline:       {gain:+.4f}")

if best_safe[0] != "greedy" and gain > 0 and best_safe[5] >= base_r - REC_EPS:
    if gain >= 0.005:
        print(f"\n>>> ADOPT {best_safe[0]}: a clear recall-safe gain "
              f"(F1 {base_f1:.4f} -> {best_safe[3]:.4f}, recall held). Ship this LM weight.")
    else:
        print(f"\n>>> MARGINAL recall-safe gain ({gain:+.4f}) at {best_safe[0]}: "
              f"strictly better (precision up, recall flat) but small. Optional to ship. "
              f"Headline result stays F1~0.667 — the LM confirms we're near the single-model "
              f"ceiling; the remaining errors are ACOUSTIC (mis-heard audio), which an LM "
              f"cannot fix.")
else:
    print(f"\n>>> KEEP GREEDY (0.667). No recall-safe setting beat baseline. Any F1 gain "
          f"came from suppressing detections (lower recall), which we refuse for an MDD task.")

### Reading this notebook
- **If a recall-safe setting beats 0.667** → adopt that `alpha/beta` as a free,
  inference-time product setting. Record it in RESULTS.md.
- **If only recall-DROPPING settings raised F1** → we KEEP greedy 0.667. That is
  the correct call: a higher F1 bought by missing real mispronunciations is a
  worse product, even if it looks better on the leaderboard. Document the
  negative result and stop here.
- The guard rail (`recall >= baseline`) is the whole point: it makes the
  over-correction risk an explicit, enforced constraint rather than a hope.

> If you later want the absolute-max leaderboard F1 regardless of recall, that's
> the `best_any` line — but do NOT ship it without understanding it hides errors.